# Gradient Descent — Theory Notebook

> *"If you want to find the minimum of a function, just follow the gradient downhill."*

Interactive demonstrations of gradient descent convergence, momentum, Nesterov acceleration, and line search methods. Companion to [notes.md](notes.md).

In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.optimize as opt

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
        HAS_SNS = True
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
        HAS_SNS = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB',
    'secondary': '#EE7733',
    'tertiary': '#009988',
    'error': '#CC3311',
    'neutral': '#555555',
    'highlight': '#EE3377',
}

if HAS_MPL:
    mpl.rcParams.update({
        'figure.figsize': (10, 6),
        'figure.dpi': 120,
        'font.size': 13,
        'axes.titlesize': 15,
        'axes.labelsize': 13,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'lines.linewidth': 2.0,
        'axes.spines.top': False,
        'axes.spines.right': False,
    })

np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)
print('Setup complete.')

---

## 1. Intuition: Gradient Descent on a 2D Quadratic

We start with the simplest non-trivial case: minimizing $f(\mathbf{x}) = \frac{1}{2}(x_1^2 + \kappa x_2^2)$ where $\kappa$ is the condition number.

In [ ]:
# === 1.1 GD on 2D quadratic ===
kappa = 10  # condition number
A = np.diag([1.0, kappa])

def f(x):
    return 0.5 * x @ A @ x

def grad_f(x):
    return A @ x

def gd_2d(x0, eta, n_steps=50):
    xs = [x0.copy()]
    x = x0.copy()
    for _ in range(n_steps):
        x = x - eta * grad_f(x)
        xs.append(x.copy())
    return np.array(xs)

x0 = np.array([1.0, 1.0])
L = kappa  # largest eigenvalue
eta_safe = 1.0 / L

xs = gd_2d(x0, eta_safe, n_steps=50)
print(f'Initial point: {x0}')
print(f'Condition number: {kappa}')
print(f'L = {L}, eta = {eta_safe}')
print(f'f(x0) = {f(x0):.6f}')
print(f'f(x_50) = {f(xs[-1]):.10f}')
print(f'Distance to optimum: {np.linalg.norm(xs[-1]):.10f}')

ok = f(xs[-1]) < 1e-6
print(f"{'PASS' if ok else 'FAIL'} — converged to near-zero after 50 steps")

In [ ]:
# === 1.2 Visualize GD trajectory ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Contour plot
    x1 = np.linspace(-1.2, 1.2, 100)
    x2 = np.linspace(-1.2, 1.2, 100)
    X1, X2 = np.meshgrid(x1, x2)
    Z = 0.5 * (X1**2 + kappa * X2**2)
    levels = np.logspace(-4, 1, 20)
    cf = ax.contour(X1, X2, Z, levels=levels, cmap='viridis', alpha=0.7)
    ax.clabel(cf, inline=True, fontsize=9, fmt='%.1e')
    
    # GD trajectory
    ax.plot(xs[:, 0], xs[:, 1], 'o-', color=COLORS['error'], 
            markersize=4, linewidth=1.5, label='GD trajectory')
    ax.plot(xs[0, 0], xs[0, 1], 's', color=COLORS['highlight'], 
            markersize=10, label='Start')
    ax.plot(0, 0, '*', color=COLORS['error'], markersize=15, label='Minimum')
    
    ax.set_title(f'GD trajectory on elliptic bowl (kappa={kappa})')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend()
    ax.set_aspect('equal')
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 2. Convergence Theory — Convex Functions

Verify the $O(1/T)$ convergence rate for convex, L-smooth functions.

In [ ]:
# === 2.1 O(1/T) convergence rate ===
def test_convex_convergence():
    kappa = 100
    A = np.diag([1.0, kappa])
    L = kappa
    eta = 1.0 / L
    x0 = np.array([1.0, 1.0])
    
    T_vals = [10, 50, 100, 500, 1000, 5000]
    errors = []
    
    x = x0.copy()
    f_star = 0.0
    
    for t in range(max(T_vals)):
        g = A @ x
        x = x - eta * g
        if (t + 1) in T_vals:
            err = 0.5 * x @ A @ x - f_star
            errors.append((t + 1, err))
    
    print('T       | f(x_T) - f*  | T * error   | Predicted bound')
    print('-' * 65)
    
    for T, err in errors:
        bound = L * np.linalg.norm(x0)**2 / (2 * T)
        print(f'{T:<7} | {err:.6e}  | {T*err:.6e}  | {bound:.6e}')
    
    # Verify: T * error should be bounded by constant
    T_last, err_last = errors[-1]
    bound_last = L * np.linalg.norm(x0)**2 / (2 * T_last)
    ok = err_last <= bound_last
    print(f"\n{'PASS' if ok else 'FAIL'} — error bounded by L*||x0||^2/(2T)")

test_convex_convergence()

In [ ]:
# === 2.2 Plot convergence rate ===
if HAS_MPL:
    kappa = 100
    A = np.diag([1.0, kappa])
    L = kappa
    eta = 1.0 / L
    x0 = np.array([1.0, 1.0])
    
    T = 2000
    x = x0.copy()
    errors = []
    
    for t in range(T):
        g = A @ x
        x = x - eta * g
        errors.append(0.5 * x @ A @ x)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    ax.plot(steps, errors, color=COLORS['primary'], label='GD error')
    ax.plot(steps, [L * np.linalg.norm(x0)**2 / (2 * t) for t in steps],
            color=COLORS['neutral'], linestyle='--', label='O(1/T) bound')
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_title('GD convergence: f(x_T) - f* vs iterations')
    ax.set_xlabel('Iteration T')
    ax.set_ylabel('Suboptimality')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 3. Convergence Theory — Strongly Convex Functions

Verify linear convergence $O((1 - \mu/L)^T)$ for strongly convex functions.

In [ ]:
# === 3.1 Linear convergence ===
kappa = 100
mu = 1.0
L = kappa
A = np.diag([mu, L])

eta_opt = 2.0 / (L + mu)  # optimal step size
eta_safe = 1.0 / L         # safe step size

x0 = np.array([1.0, 1.0])
T = 500

def run_gd(A, x0, eta, T):
    x = x0.copy()
    errors = []
    for _ in range(T):
        g = A @ x
        x = x - eta * g
        errors.append(0.5 * x @ A @ x)
    return np.array(errors)

err_opt = run_gd(A, x0, eta_opt, T)
err_safe = run_gd(A, x0, eta_safe, T)

rate_opt = (1 - mu/L)
rate_safe = (1 - 1/kappa)

print(f'Condition number: kappa = {kappa}')
print(f'Optimal eta = {eta_opt:.6f}, rate = {rate_opt:.6f}')
print(f'Safe eta = {eta_safe:.6f}, rate = {rate_safe:.6f}')
print()
print(f'After {T} steps:')
print(f'  Optimal eta: error = {err_opt[-1]:.2e}')
print(f'  Safe eta:    error = {err_safe[-1]:.2e}')

ok = err_opt[-1] < err_safe[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — optimal eta converges faster")

In [ ]:
# === 3.2 Plot linear convergence ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    
    ax.plot(steps, err_opt, color=COLORS['primary'], label=f'Optimal eta={eta_opt:.4f}')
    ax.plot(steps, err_safe, color=COLORS['secondary'], label=f'Safe eta={eta_safe:.4f}')
    
    # Theoretical bounds
    f0 = 0.5 * x0 @ A @ x0
    ax.plot(steps, f0 * (1 - mu/L)**steps, color=COLORS['primary'],
            linestyle='--', alpha=0.5, label='Theoretical rate (optimal)')
    
    ax.set_yscale('log')
    ax.set_title('Linear convergence for strongly convex functions')
    ax.set_xlabel('Iteration T')
    ax.set_ylabel('Suboptimality (log scale)')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 4. Non-Convex Convergence

GD on non-convex functions converges to stationary points with gradient norm decay $O(1/\sqrt{T})$.

In [ ]:
# === 4.1 Non-convex function: Rosenbrock ===
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def rosenbrock_grad(x):
    dx = -2*(1 - x[0]) - 400*x[0]*(x[1] - x[0]**2)
    dy = 200*(x[1] - x[0]**2)
    return np.array([dx, dy])

x0 = np.array([0.0, 0.0])
eta = 0.0005
T = 50000

x = x0.copy()
grad_norms = []
f_vals = []

for t in range(T):
    g = rosenbrock_grad(x)
    x = x - eta * g
    grad_norms.append(np.linalg.norm(g))
    f_vals.append(rosenbrock(x))

grad_norms = np.array(grad_norms)
f_vals = np.array(f_vals)

print(f'Rosenbrock function: f*(1,1) = 0')
print(f'Start: {x0}, f(x0) = {rosenbrock(x0):.2f}')
print(f'After {T} steps: x = ({x[0]:.4f}, {x[1]:.4f})')
print(f'f(x_T) = {f_vals[-1]:.6f}')
print(f'Final gradient norm: {grad_norms[-1]:.6f}')
print(f'Min gradient norm: {grad_norms.min():.6f}')

ok = grad_norms[-1] < 0.1
print(f"\n{'PASS' if ok else 'FAIL'} — gradient norm decreased")

In [ ]:
# === 4.2 Plot gradient norm decay ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    steps = np.arange(1, T + 1)
    
    axes[0].plot(steps, f_vals, color=COLORS['primary'])
    axes[0].set_title('Function value over iterations')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('f(x)')
    axes[0].set_yscale('log')
    
    axes[1].plot(steps, grad_norms, color=COLORS['error'])
    axes[1].axhline(0.1, color=COLORS['neutral'], linestyle='--', label='threshold')
    axes[1].set_title('Gradient norm over iterations')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('||grad f(x)||')
    axes[1].set_yscale('log')
    axes[1].legend()
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 5. Momentum

Polyak momentum accelerates convergence on ill-conditioned problems by accumulating velocity in consistent descent directions.

In [ ]:
# === 5.1 Momentum vs vanilla GD ===
kappa = 100
mu = 1.0
L = kappa
A = np.diag([mu, L])
x0 = np.array([1.0, 1.0])
T = 200

# Vanilla GD
eta_gd = 1.0 / L
x = x0.copy()
err_gd = []
for _ in range(T):
    x = x - eta_gd * (A @ x)
    err_gd.append(0.5 * x @ A @ x)

# Momentum with optimal beta
beta_opt = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1))**2
eta_mom = 1.0 / L
x = x0.copy()
v = np.zeros_like(x)
err_mom = []
for _ in range(T):
    v = beta_opt * v + A @ x
    x = x - eta_mom * v
    err_mom.append(0.5 * x @ A @ x)

# Momentum with default beta=0.9
beta_default = 0.9
x = x0.copy()
v = np.zeros_like(x)
err_mom_default = []
for _ in range(T):
    v = beta_default * v + A @ x
    x = x - eta_mom * v
    err_mom_default.append(0.5 * x @ A @ x)

err_gd = np.array(err_gd)
err_mom = np.array(err_mom)
err_mom_default = np.array(err_mom_default)

print(f'Condition number: kappa = {kappa}')
print(f'Optimal beta = {beta_opt:.4f}')
print()
print(f'After {T} iterations:')
print(f'  Vanilla GD:       error = {err_gd[-1]:.2e}')
print(f'  Momentum (opt):   error = {err_mom[-1]:.2e}')
print(f'  Momentum (0.9):   error = {err_mom_default[-1]:.2e}')
print()
speedup = err_gd[-1] / max(err_mom[-1], 1e-30)
print(f'Momentum speedup: {speedup:.1f}x')

ok = err_mom[-1] < err_gd[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — momentum converges faster")

In [ ]:
# === 5.2 Plot momentum comparison ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    
    ax.plot(steps, err_gd, color=COLORS['neutral'], label='Vanilla GD')
    ax.plot(steps, err_mom, color=COLORS['primary'], label=f'Momentum (beta={beta_opt:.3f})')
    ax.plot(steps, err_mom_default, color=COLORS['secondary'], label='Momentum (beta=0.9)')
    
    ax.set_yscale('log')
    ax.set_title('Momentum accelerates convergence on ill-conditioned problems')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Suboptimality (log scale)')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 6. Nesterov Accelerated Gradient

NAG achieves the optimal $O(1/T^2)$ rate for convex functions.

In [ ]:
# === 6.1 NAG vs GD vs Momentum ===
kappa = 100
A = np.diag([1.0, kappa])
L = kappa
x0 = np.array([1.0, 1.0])
T = 500

# Vanilla GD
eta = 1.0 / L
x = x0.copy()
err_gd = []
for _ in range(T):
    x = x - eta * (A @ x)
    err_gd.append(0.5 * x @ A @ x)

# NAG (FISTA-style)
y = x0.copy()
x_prev = x0.copy()
t = 1.0
err_nag = []
for _ in range(T):
    x_new = y - eta * (A @ y)
    t_new = (1 + np.sqrt(1 + 4*t**2)) / 2
    y = x_new + ((t - 1) / t_new) * (x_new - x_prev)
    x_prev = x_new
    t = t_new
    err_nag.append(0.5 * x_new @ A @ x_new)

err_gd = np.array(err_gd)
err_nag = np.array(err_nag)

print(f'Condition number: kappa = {kappa}')
print(f'After {T} iterations:')
print(f'  Vanilla GD: error = {err_gd[-1]:.2e}')
print(f'  NAG:        error = {err_nag[-1]:.2e}')
print()
print(f'Theoretical GD bound: O(1/T) = {L * np.linalg.norm(x0)**2 / (2*T):.2e}')
print(f'Theoretical NAG bound: O(1/T^2) = {2*L*np.linalg.norm(x0)**2 / (T+1)**2:.2e}')

ok = err_nag[-1] < err_gd[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — NAG outperforms GD")

In [ ]:
# === 6.2 Plot NAG vs GD ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    
    ax.plot(steps, err_gd, color=COLORS['neutral'], label='Vanilla GD')
    ax.plot(steps, err_nag, color=COLORS['primary'], label='Nesterov AGD')
    ax.plot(steps, [L * np.linalg.norm(x0)**2 / (2*t) for t in steps],
            color=COLORS['neutral'], linestyle='--', alpha=0.5, label='O(1/T) bound')
    ax.plot(steps, [2*L*np.linalg.norm(x0)**2 / (t+1)**2 for t in steps],
            color=COLORS['primary'], linestyle='--', alpha=0.5, label='O(1/T^2) bound')
    
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_title('Nesterov acceleration: O(1/T^2) vs O(1/T)')
    ax.set_xlabel('Iteration T')
    ax.set_ylabel('Suboptimality')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 7. Line Search Methods

Armijo backtracking automatically selects a step size that guarantees sufficient decrease.

In [ ]:
# === 7.1 Armijo backtracking ===
def armijo_backtrack(f, grad_f, x, g, eta_bar=1.0, rho=0.5, c=1e-4, max_iter=50):
    eta = eta_bar
    f_x = f(x)
    slope = -np.dot(g, g)
    for _ in range(max_iter):
        if f(x + eta * (-g)) <= f_x + c * eta * slope:
            return eta
        eta *= rho
    return eta

# Test on 2D quadratic
kappa = 100
A = np.diag([1.0, kappa])

def f(x): return 0.5 * x @ A @ x
def grad_f(x): return A @ x

x = np.array([1.0, 1.0])
etap_history = []

for t in range(20):
    g = grad_f(x)
    eta = armijo_backtrack(f, grad_f, x, g, eta_bar=1.0)
    etap_history.append(eta)
    x = x - eta * g

print('Armijo backtracking step sizes:')
for i, eta in enumerate(etap_history):
    print(f'  Step {i+1:2d}: eta = {eta:.6f}, f(x) = {f(x):.10f}')

ok = f(x) < 1e-6
print(f"\n{'PASS' if ok else 'FAIL'} — Armijo backtracking converged")

In [ ]:
# === 7.2 Visualize Armijo condition ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot step sizes over iterations
    ax.plot(range(1, len(etap_history)+1), etap_history, 
            'o-', color=COLORS['primary'], markersize=6)
    ax.axhline(1.0/kappa, color=COLORS['neutral'], linestyle='--',
               label=f'1/L = {1.0/kappa:.4f}')
    ax.set_title('Armijo backtracking step sizes')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Step size eta')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 8. Condition Number Effects on Convergence

The condition number $\kappa = L/\mu$ is the single most important factor controlling GD convergence speed.

In [ ]:
# === 8.1 Convergence vs condition number ===
kappa_values = [1, 5, 10, 50, 100, 500, 1000]
T = 1000
x0 = np.array([1.0, 1.0])

results = []
for kappa in kappa_values:
    A = np.diag([1.0, kappa])
    L = kappa
    eta = 1.0 / L
    x = x0.copy()
    for _ in range(T):
        x = x - eta * (A @ x)
    err = 0.5 * x @ A @ x
    
    # Find iterations to reach 1e-6
    x = x0.copy()
    iters_needed = 0
    for t in range(100000):
        x = x - eta * (A @ x)
        if 0.5 * x @ A @ x < 1e-6:
            iters_needed = t + 1
            break
    else:
        iters_needed = -1  # didn't converge
    
    results.append((kappa, err, iters_needed))

print(f'kappa  | Error after {T} iters | Iters to 1e-6')
print('-' * 50)
for kappa, err, iters in results:
    iters_str = f'{iters}' if iters > 0 else 'did not converge'
    print(f'{kappa:<6} | {err:.2e}        | {iters_str}')

# Verify O(kappa) scaling
iters_10 = results[2][2]  # kappa=10
iters_100 = results[4][2]  # kappa=100
if iters_10 > 0 and iters_100 > 0:
    ratio = iters_100 / iters_10
    print(f'\nRatio iters(kappa=100)/iters(kappa=10) = {ratio:.1f} (expected ~10)')
    ok = 5 < ratio < 20
    print(f"{'PASS' if ok else 'FAIL'} — iterations scale linearly with kappa")
else:
    print('Could not verify scaling (one did not converge)')

In [ ]:
# === 8.2 Plot condition number effect ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: convergence curves for different kappa
    for kappa in [1, 10, 100, 1000]:
        A = np.diag([1.0, kappa])
        L = kappa
        eta = 1.0 / L
        x = x0.copy()
        errs = []
        for _ in range(T):
            x = x - eta * (A @ x)
            errs.append(0.5 * x @ A @ x)
        axes[0].plot(range(1, T+1), errs, label=f'kappa={kappa}')
    
    axes[0].set_yscale('log')
    axes[0].set_title('Convergence for different condition numbers')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Suboptimality')
    axes[0].legend()
    
    # Right: iterations to converge vs kappa
    kappas = [r[0] for r in results if r[2] > 0]
    iters_list = [r[2] for r in results if r[2] > 0]
    axes[1].plot(kappas, iters_list, 'o-', color=COLORS['primary'])
    axes[1].plot(kappas, [iters_list[0] * k / kappas[0] for k in kappas],
                 '--', color=COLORS['neutral'], label='O(kappa) trend')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_title('Iterations to converge vs condition number')
    axes[1].set_xlabel('Condition number kappa')
    axes[1].set_ylabel('Iterations')
    axes[1].legend()
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 9. GD for Linear Regression

Apply gradient descent to ordinary least squares and compare with the closed-form solution.

In [ ]:
# === 9.1 Linear regression with GD ===
n, d = 200, 10
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = X @ w_true + 0.1 * np.random.randn(n)

# Closed-form solution
w_star = np.linalg.lstsq(X, y, rcond=None)[0]
loss_star = 0.5 * np.mean((X @ w_star - y)**2)

# GD solution
L = np.linalg.norm(X.T @ X) / n  # smoothness constant
eta = 1.0 / L
w = np.zeros(d)
T = 5000
loss_history = []

for _ in range(T):
    grad = X.T @ (X @ w - y) / n
    w = w - eta * grad
    loss_history.append(0.5 * np.mean((X @ w - y)**2))

loss_gd = loss_history[-1]

print(f'Problem: n={n}, d={d}')
print(f'Condition number of X^T X: {np.linalg.cond(X.T @ X):.1f}')
print(f'Smoothness constant L: {L:.4f}')
print(f'Learning rate eta: {eta:.6f}')
print()
print(f'Closed-form loss: {loss_star:.8f}')
print(f'GD loss after {T} steps: {loss_gd:.8f}')
print(f'Weight error ||w_gd - w_star||: {np.linalg.norm(w - w_star):.8f}')

ok = np.allclose(w, w_star, atol=1e-5)
print(f"\n{'PASS' if ok else 'FAIL'} — GD matches closed-form solution")

---

## 10. PL (Polyak-Łojasiewicz) Condition

The PL condition gives linear convergence without convexity.

In [ ]:
# === 10.1 PL condition verification ===
# Function: f(x) = (1 - exp(-x^2))^2
# This is non-convex but satisfies the PL condition

def f_pl(x):
    return (1 - np.exp(-x**2))**2

def grad_pl(x):
    return 4 * x * np.exp(-x**2) * (1 - np.exp(-x**2))

# Verify PL condition: 0.5 * |f'(x)|^2 >= mu * (f(x) - f*)
x_test = np.linspace(-3, 3, 1000)
f_vals = f_pl(x_test)
g_vals = grad_pl(x_test)
f_star = 0.0  # minimum at x=0

# Find largest mu such that 0.5 * |g|^2 >= mu * f for all x
ratios = np.where(f_vals > 1e-10, 0.5 * g_vals**2 / f_vals, np.inf)
mu_pl = np.min(ratios[ratios < np.inf])

print(f'PL constant mu = {mu_pl:.6f}')
print(f'PL condition holds: 0.5 * |grad f|^2 >= {mu_pl:.4f} * f for all x')

# GD with PL condition
eta = 0.5
x = 2.0
T = 100
err_pl = []
for _ in range(T):
    g = grad_pl(x)
    x = x - eta * g
    err_pl.append(f_pl(x))

err_pl = np.array(err_pl)
print(f'\nAfter {T} iterations: f(x) = {err_pl[-1]:.2e}')

# Check linear convergence
if err_pl[0] > 0 and err_pl[-1] > 0:
    empirical_rate = (err_pl[-1] / err_pl[0])**(1/T)
    print(f'Empirical convergence rate: {empirical_rate:.4f}')
    print(f'Theoretical rate bound: {1 - mu_pl * eta:.4f}')

ok = err_pl[-1] < 1e-6
print(f"\n{'PASS' if ok else 'FAIL'} — GD converges linearly under PL condition")

In [ ]:
# === 10.2 Plot PL function and convergence ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Function plot
    axes[0].plot(x_test, f_vals, color=COLORS['primary'])
    axes[0].set_title('PL function: f(x) = (1 - exp(-x^2))^2')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].axvline(0, color=COLORS['neutral'], linestyle='--', alpha=0.5)
    
    # Convergence
    axes[1].plot(range(1, T+1), err_pl, color=COLORS['primary'])
    axes[1].set_yscale('log')
    axes[1].set_title('Linear convergence under PL condition')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('f(x)')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## Summary

This notebook demonstrated:

1. **GD on 2D quadratics** — visualizing the zigzag pattern on ill-conditioned problems
2. **Convex convergence** — verifying the $O(1/T)$ rate
3. **Strongly convex convergence** — linear convergence $O((1-\mu/L)^T)$
4. **Non-convex convergence** — gradient norm decay $O(1/\sqrt{T})$ on the Rosenbrock function
5. **Momentum** — acceleration on ill-conditioned problems, optimal $\beta$
6. **Nesterov AGD** — $O(1/T^2)$ rate matching the lower bound
7. **Armijo backtracking** — automatic step size selection
8. **Condition number effects** — iterations scale as $O(\kappa)$
9. **Linear regression** — GD matches the closed-form solution
10. **PL condition** — linear convergence without convexity

See [exercises.ipynb](exercises.ipynb) for graded problems on these topics.

---

## 11. Momentum Trajectory Visualization

Compare the paths taken by vanilla GD and momentum on an ill-conditioned quadratic.

In [ ]:
# === 11.1 GD vs momentum trajectories ===
kappa = 20
A = np.diag([1.0, kappa])
L = kappa
eta = 1.0 / L
x0 = np.array([1.0, 1.0])
T = 30

# Vanilla GD
x_gd = x0.copy()
path_gd = [x_gd.copy()]
for _ in range(T):
    x_gd = x_gd - eta * (A @ x_gd)
    path_gd.append(x_gd.copy())
path_gd = np.array(path_gd)

# Momentum
beta = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1))**2
x_mom = x0.copy()
v = np.zeros(2)
path_mom = [x_mom.copy()]
for _ in range(T):
    v = beta * v + A @ x_mom
    x_mom = x_mom - eta * v
    path_mom.append(x_mom.copy())
path_mom = np.array(path_mom)

print(f'Condition number: {kappa}')
print(f'Optimal beta: {beta:.4f}')
print(f'GD final error: {0.5 * path_gd[-1] @ A @ path_gd[-1]:.2e}')
print(f'Momentum final error: {0.5 * path_mom[-1] @ A @ path_mom[-1]:.2e}')

In [ ]:
# === 11.2 Plot trajectories ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 7))
    
    x1 = np.linspace(-1.2, 1.2, 100)
    x2 = np.linspace(-1.2, 1.2, 100)
    X1, X2 = np.meshgrid(x1, x2)
    Z = 0.5 * (X1**2 + kappa * X2**2)
    levels = np.logspace(-4, 1, 15)
    cf = ax.contour(X1, X2, Z, levels=levels, cmap='viridis', alpha=0.6)
    
    ax.plot(path_gd[:, 0], path_gd[:, 1], 'o-', color=COLORS['neutral'],
            markersize=3, linewidth=1, label='Vanilla GD', alpha=0.7)
    ax.plot(path_mom[:, 0], path_mom[:, 1], 's-', color=COLORS['primary'],
            markersize=4, linewidth=1.5, label='Momentum')
    ax.plot(x0[0], x0[1], '*', color=COLORS['highlight'], markersize=12, label='Start')
    ax.plot(0, 0, 'r*', markersize=15, label='Minimum')
    
    ax.set_title(f'GD vs Momentum trajectories (kappa={kappa})')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend()
    ax.set_aspect('equal')
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 12. Step Size Effects

Explore what happens when the learning rate is too small, optimal, or too large.

In [ ]:
# === 12.1 Different step sizes ===
kappa = 10
A = np.diag([1.0, kappa])
L = kappa
x0 = np.array([1.0, 1.0])
T = 50

etas = [0.5/L, 1.0/L, 1.5/L, 1.9/L, 2.0/L, 2.1/L]
all_errors = {}

for eta in etas:
    x = x0.copy()
    errs = []
    diverged = False
    for t in range(T):
        x = x - eta * (A @ x)
        err = 0.5 * x @ A @ x
        if err > 1e10:
            diverged = True
            break
        errs.append(err)
    all_errors[eta] = (np.array(errs), diverged)

print('Step size effects on convergence:')
print(f'L = {L}, stability boundary = 2/L = {2/L:.4f}')
print()
for eta, (errs, div) in all_errors.items():
    status = 'DIVERGED' if div else f'error={errs[-1]:.2e}'
    print(f'eta = {eta:.4f} ({eta*L:.1f}/L): {status}')

ok = not all_errors[2.1/L][1]  # should diverge
print(f"\n{'FAIL' if not ok else 'PASS'} — eta > 2/L causes divergence")

In [ ]:
# === 12.2 Plot step size comparison ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for eta, (errs, div) in all_errors.items():
        label = f'eta={eta*L:.1f}/L'
        if div:
            ax.plot(range(len(errs)), errs, '--', alpha=0.5, label=label)
        else:
            ax.plot(range(len(errs)), errs, label=label)
    
    ax.axvline(0, color='black', alpha=0.1)
    ax.set_yscale('log')
    ax.set_title('Effect of step size on convergence')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Suboptimality')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 13. NAG on Logistic Regression

Compare GD, momentum, and NAG on a real ML loss function.

In [ ]:
# === 13.1 Logistic regression setup ===
np.random.seed(42)
n, d = 500, 5
X = np.random.randn(n, d)
w_true = np.array([1.0, -0.5, 0.8, -0.3, 0.6])
y = (X @ w_true + np.random.randn(n) * 0.5 > 0).astype(float) * 2 - 1

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def log_loss(w):
    z = y * (X @ w)
    return np.mean(np.log(1 + np.exp(-np.clip(z, -500, 500))))

def log_loss_grad(w):
    z = y * (X @ w)
    s = sigmoid(-z)
    return -X.T @ (y * s) / n

# Estimate L
L_est = np.linalg.norm(X.T @ X) / (4 * n)
print(f'Logistic regression: n={n}, d={d}')
print(f'Estimated smoothness constant L = {L_est:.6f}')
print(f'Optimal learning rate eta = 1/L = {1/L_est:.6f}')

In [ ]:
# === 13.2 Compare optimizers on logistic regression ===
eta = 1.0 / L_est
T = 500
w0 = np.zeros(d)

# Vanilla GD
w = w0.copy()
err_gd = []
for _ in range(T):
    w = w - eta * log_loss_grad(w)
    err_gd.append(log_loss(w))

# Momentum
beta = 0.9
w = w0.copy()
v = np.zeros(d)
err_mom = []
for _ in range(T):
    v = beta * v + log_loss_grad(w)
    w = w - eta * v
    err_mom.append(log_loss(w))

# NAG
w = w0.copy()
y_vec = w0.copy()
w_prev = w0.copy()
t_val = 1.0
err_nag = []
for _ in range(T):
    w_new = y_vec - eta * log_loss_grad(y_vec)
    t_new = (1 + np.sqrt(1 + 4*t_val**2)) / 2
    y_vec = w_new + ((t_val - 1) / t_new) * (w_new - w_prev)
    w_prev = w_new
    t_val = t_new
    err_nag.append(log_loss(w_new))

err_gd = np.array(err_gd)
err_mom = np.array(err_mom)
err_nag = np.array(err_nag)

print(f'After {T} iterations:')
print(f'  GD:     loss = {err_gd[-1]:.6f}')
print(f'  Mom:    loss = {err_mom[-1]:.6f}')
print(f'  NAG:    loss = {err_nag[-1]:.6f}')
print()
print(f'Best loss: {min(err_gd[-1], err_mom[-1], err_nag[-1]):.6f}')

ok = min(err_mom[-1], err_nag[-1]) < err_gd[-1]
print(f"\n{'PASS' if ok else 'FAIL'} — accelerated methods outperform GD")

In [ ]:
# === 13.3 Plot logistic regression convergence ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    f_star = min(err_gd[-1], err_mom[-1], err_nag[-1])
    
    ax.plot(steps, err_gd - f_star, color=COLORS['neutral'], label='GD')
    ax.plot(steps, err_mom - f_star, color=COLORS['secondary'], label='Momentum')
    ax.plot(steps, err_nag - f_star, color=COLORS['primary'], label='NAG')
    
    ax.set_yscale('log')
    ax.set_title('Optimizer comparison on logistic regression')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss - f* (log scale)')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 14. Heavy Ball Dynamics

Visualize how momentum creates oscillatory behavior that dampens over time.

In [ ]:
# === 14.1 Velocity magnitude over time ===
kappa = 50
A = np.diag([1.0, kappa])
L = kappa
eta = 1.0 / L
beta = 0.9
x0 = np.array([1.0, 1.0])
T = 100

x = x0.copy()
v = np.zeros(2)
vel_norms = []
err_mom = []

for _ in range(T):
    g = A @ x
    v = beta * v + g
    x = x - eta * v
    vel_norms.append(np.linalg.norm(v))
    err_mom.append(0.5 * x @ A @ x)

vel_norms = np.array(vel_norms)
err_mom = np.array(err_mom)

print(f'Heavy ball method: kappa={kappa}, beta={beta}')
print(f'Max velocity norm: {vel_norms.max():.4f}')
print(f'Final velocity norm: {vel_norms[-1]:.6f}')
print(f'Final error: {err_mom[-1]:.2e}')

ok = vel_norms[-1] < vel_norms.max() * 0.01
print(f"\n{'PASS' if ok else 'FAIL'} — velocity dampens to near zero")

In [ ]:
# === 14.2 Plot velocity and error ===
if HAS_MPL:
    fig, axes = plt.subplots(2, 1, figsize=(10, 10))
    steps = np.arange(1, T + 1)
    
    axes[0].plot(steps, err_mom, color=COLORS['primary'])
    axes[0].set_yscale('log')
    axes[0].set_title('Error over iterations (momentum)')
    axes[0].set_ylabel('Suboptimality')
    
    axes[1].plot(steps, vel_norms, color=COLORS['secondary'])
    axes[1].set_title('Velocity magnitude over iterations')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('||v_t||')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 15. Implicit Bias of Gradient Descent

In overparameterized settings, GD converges to the minimum-norm solution.

In [ ]:
# === 15.1 Overparameterized linear system ===
np.random.seed(42)
n, d = 50, 200  # underdetermined: more parameters than equations
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = X @ w_true

# Minimum norm solution via pseudoinverse
w_min_norm = X.T @ np.linalg.lstsq(X @ X.T, y, rcond=None)[0]

# GD from zero initialization
L = np.linalg.norm(X.T @ X)
eta = 1.0 / L
w = np.zeros(d)
T = 10000

for _ in range(T):
    grad = X.T @ (X @ w - y)
    w = w - eta * grad

print(f'Overparameterized system: n={n}, d={d}')
print(f'Condition number of XX^T: {np.linalg.cond(X @ X.T):.1f}')
print()
print(f'Min-norm solution ||w*||: {np.linalg.norm(w_min_norm):.6f}')
print(f'GD solution ||w_gd||: {np.linalg.norm(w):.6f}')
print(f'||w_gd - w*||: {np.linalg.norm(w - w_min_norm):.6e}')
print(f'Residual ||Xw - y||: {np.linalg.norm(X @ w - y):.2e}')

ok = np.linalg.norm(w - w_min_norm) < 1e-4
print(f"\n{'PASS' if ok else 'FAIL'} — GD converges to minimum-norm solution")

---

## 16. Gradient Clipping

Demonstrate how gradient clipping prevents divergence on ill-conditioned problems.

In [ ]:
# === 16.1 GD with and without clipping ===
kappa = 100
A = np.diag([1.0, kappa])
L = kappa
x0 = np.array([1.0, 1.0])
T = 50

# GD without clipping (large eta)
eta_large = 2.5 / L  # beyond stability boundary
x = x0.copy()
err_no_clip = []
grad_norms_no = []
for _ in range(T):
    g = A @ x
    grad_norms_no.append(np.linalg.norm(g))
    x = x - eta_large * g
    err = 0.5 * x @ A @ x
    if err > 1e15:
        break
    err_no_clip.append(err)

# GD with clipping
max_norm = 5.0
x = x0.copy()
err_clip = []
grad_norms_clip = []
for _ in range(T):
    g = A @ x
    g_norm = np.linalg.norm(g)
    if g_norm > max_norm:
        g = g * (max_norm / g_norm)
    grad_norms_clip.append(np.linalg.norm(g))
    x = x - eta_large * g
    err_clip.append(0.5 * x @ A @ x)

print(f'Large eta = {eta_large:.4f} (> 2/L = {2/L:.4f})')
print(f'Gradient clip threshold: {max_norm}')
print()
print(f'Without clipping: diverged after {len(err_no_clip)} steps')
print(f'With clipping: final error = {err_clip[-1]:.2e}')

ok = err_clip[-1] < 1.0
print(f"\n{'PASS' if ok else 'FAIL'} — clipping prevents divergence")

In [ ]:
# === 16.2 Plot clipping effect ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(range(1, len(err_no_clip)+1), err_no_clip,
                 color=COLORS['error'], label='No clipping')
    axes[0].plot(range(1, T+1), err_clip, color=COLORS['primary'], label='With clipping')
    axes[0].set_yscale('log')
    axes[0].set_title('Gradient clipping prevents divergence')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Suboptimality')
    axes[0].legend()
    
    axes[1].plot(range(1, len(grad_norms_no)+1), grad_norms_no,
                 color=COLORS['error'], label='No clipping')
    axes[1].plot(range(1, T+1), grad_norms_clip, color=COLORS['primary'], label='With clipping')
    axes[1].axhline(max_norm, color=COLORS['neutral'], linestyle='--', label='Clip threshold')
    axes[1].set_title('Gradient norms')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('||gradient||')
    axes[1].legend()
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 17. Summary: All Methods Compared

Final comparison of all gradient descent variants on a standard test problem.

In [ ]:
# === 17.1 Comprehensive comparison ===
kappa = 100
A = np.diag([1.0, kappa])
L = kappa
mu = 1.0
x0 = np.array([1.0, 1.0])
T = 500
target = 1e-8

methods = {}

# 1. Vanilla GD
x = x0.copy()
for t in range(T):
    x = x - (1/L) * (A @ x)
    if 0.5 * x @ A @ x < target:
        methods['GD'] = t + 1
        break

# 2. GD with optimal eta
x = x0.copy()
for t in range(T):
    x = x - (2/(L+mu)) * (A @ x)
    if 0.5 * x @ A @ x < target:
        methods['GD (opt eta)'] = t + 1
        break

# 3. Momentum
beta_opt = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1))**2
x = x0.copy()
v = np.zeros(2)
for t in range(T):
    v = beta_opt * v + A @ x
    x = x - (1/L) * v
    if 0.5 * x @ A @ x < target:
        methods['Momentum'] = t + 1
        break

# 4. NAG
y = x0.copy()
x_prev = x0.copy()
t_val = 1.0
for t in range(T):
    x_new = y - (1/L) * (A @ y)
    t_new = (1 + np.sqrt(1 + 4*t_val**2)) / 2
    y = x_new + ((t_val - 1) / t_new) * (x_new - x_prev)
    x_prev = x_new
    t_val = t_new
    if 0.5 * x_new @ A @ x_new < target:
        methods['NAG'] = t + 1
        break

print(f'Test problem: f(x) = 0.5(x_1^2 + {kappa}*x_2^2)')
print(f'Condition number: kappa = {kappa}')
print(f'Target accuracy: {target}')
print()
print(f'Method            | Iterations')
print('-' * 35)
for name, iters in sorted(methods.items(), key=lambda x: x[1]):
    print(f'{name:<20}| {iters}')

if len(methods) >= 2:
    fastest = min(methods.values())
    slowest = max(methods.values())
    print(f'\nSpeedup (slowest/fastest): {slowest/fastest:.1f}x')
    ok = methods.get('NAG', float('inf')) <= methods.get('GD', float('inf'))
    print(f"{'PASS' if ok else 'FAIL'} — accelerated methods are faster")